In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
# === 0. Setup ===
import os
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict
from datetime import timedelta

# Optional (for torch Dataset skeleton – 학습 단계에서 유용)
try:
    import torch
    from torch.utils.data import Dataset
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False
    print("[Info] PyTorch not installed. You can still run up to fold-splitting.")

@dataclass
class Config:
    lookback: int = 28   # L
    horizon: int  = 7    # H
    patch_len: int = 7   # for PatchTST later
    stride: int    = 1   # for PatchTST later
    n_splits: int  = 5   # K-fold
    embargo_days: int = 35  # purge gap around validation (≈ lookback + horizon)
    seed: int = 42

CFG = Config()
np.random.seed(CFG.seed)

print(CFG)


Config(lookback=28, horizon=7, patch_len=7, stride=1, n_splits=5, embargo_days=35, seed=42)


In [ ]:
# === 1. Load & sort ===
# 경로는 ipynb 기준으로 맞춰주세요.
TRAIN_PATH = "./data_filtering/filtered/train.csv"  # ex) "./dataset/train/train.csv"

df = pd.read_csv(TRAIN_PATH)

# Basic parsing
df['date'] = pd.to_datetime(df['date'])
# 안전장치: store_menu 없으면 생성
if 'store_menu' not in df.columns:
    df['store_menu'] = df['store'].astype(str) + "_" + df['menu'].astype(str)

# 정렬 & 타입 정리
df = df.sort_values(['store_menu', 'date']).reset_index(drop=True)

print("Rows:", len(df))
print("Unique store_menu:", df['store_menu'].nunique())
print(df.head(3))


FileNotFoundError: [Errno 2] No such file or directory: './data_filtering/filtered/train.csv'

In [4]:
# === 2. Feature engineering (calendar only, rule-safe) ===
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()
    f['dow'] = f['date'].dt.weekday           # 0..6
    f['dom'] = f['date'].dt.day               # 1..31
    f['month'] = f['date'].dt.month           # 1..12
    f['is_weekend'] = (f['dow'] >= 5).astype(int)

    # Sine/Cos encoding for weekly seasonality
    f['dow_sin'] = np.sin(2 * np.pi * f['dow'] / 7)
    f['dow_cos'] = np.cos(2 * np.pi * f['dow'] / 7)
    return f

df_feat = add_calendar_features(df)

# 사용할 입력 피처 목록 (sales + calendar)
FEATURE_COLS = ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']
TARGET_COL = 'sales'

print("Feature columns:", FEATURE_COLS)
df_feat.head(3)


Feature columns: ['sales', 'dow', 'dom', 'month', 'is_weekend', 'dow_sin', 'dow_cos']


,date_ordinal,date,store_menu,store,menu,sales,dow,dom,month,is_weekend,dow_sin,dow_cos
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,6,1,1,1,-0.781831,0.62349
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,0,2,1,0,0.000000,1.00000
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0,1,3,1,0,0.781831,0.62349


In [5]:
# === 3. Sliding windows (28 -> 7) indexing ===
def build_samples_index(
    frame: pd.DataFrame,
    lookback: int,
    horizon: int,
    feature_cols: List[str],
    target_col: str = 'sales'
) -> pd.DataFrame:
    
    rows = []
    sample_id = 0

    for sm, g in frame.groupby('store_menu', sort=False):
        g = g.sort_values('date').reset_index(drop=True)
        n = len(g)
        # last valid target start index (inclusive)
        last_t = n - horizon
        # first valid target start index (input must fully exist)
        first_t = lookback
        for t in range(first_t, last_t + 1):
            inp_start = t - lookback
            inp_end   = t - 1
            tgt_start = t
            tgt_end   = t + horizon - 1

            rows.append({
                'sample_id': sample_id,
                'store_menu': sm,
                'input_start_idx': inp_start,
                'target_start_idx': tgt_start,
                'input_start_date': g.loc[inp_start, 'date'],
                'input_end_date':   g.loc[inp_end,   'date'],
                'target_start_date':g.loc[tgt_start, 'date'],
                'target_end_date':  g.loc[tgt_end,   'date'],
            })
            sample_id += 1

    samples = pd.DataFrame(rows).sort_values(['store_menu','target_start_date']).reset_index(drop=True)
    return samples

# 고유 ID 매핑
SM_LIST = df_feat['store_menu'].drop_duplicates().tolist()
SM2ID = {sm:i+1 for i, sm in enumerate(SM_LIST)}  # 0은 UNK
UNK_ID = 0

samples = build_samples_index(df_feat, CFG.lookback, CFG.horizon, FEATURE_COLS, TARGET_COL)
print("Total samples:", len(samples))
samples.head(3)

def future_cal_features(start_date, H=7):
    dates = pd.date_range(start_date, periods=H, freq='D')
    dow = dates.weekday
    dow_sin = np.sin(2*np.pi*dow/7)
    dow_cos = np.cos(2*np.pi*dow/7)
    is_weekend = (dow>=5).astype(int)
    # H×3 -> 평평하게(간단)
    return np.concatenate([dow_sin, dow_cos, is_weekend], axis=0).astype(np.float32)  # len=H*3



# Optional: PyTorch dataset skeleton for later training
if TORCH_AVAILABLE:
    class SalesWindowDataset(Dataset):
        def __init__(self, base_df, samples_df, feature_cols, target_col, lookback, horizon):
            self.base = base_df
            self.samples = samples_df.reset_index(drop=True)
            self.feat_cols = feature_cols
            self.tgt_col = target_col
            self.L, self.H = lookback, horizon
            self.group = {sm:g.reset_index(drop=True) for sm,g in self.base.groupby('store_menu', sort=False)}

        def __len__(self): return len(self.samples)

        def __getitem__(self, idx):
            s = self.samples.iloc[idx]
            g = self.group[s['store_menu']]
            inp_start = int(s['input_start_idx']); inp_end = inp_start + self.L
            tgt_start = int(s['target_start_idx']); tgt_end = tgt_start + self.H

            X = g.loc[inp_start:inp_end-1, self.feat_cols].to_numpy(np.float32)   # (L,C)
            y = g.loc[tgt_start:tgt_end-1, self.tgt_col].to_numpy(np.float32)     # (H,)

            # store_menu id
            sm_id = SM2ID.get(s['store_menu'], UNK_ID)

            # 타깃 시작일 기준 미래 달력 피처
            fut_cal = future_cal_features(s['target_start_date'], H=self.H)        # (H*3,)

            return torch.from_numpy(X), torch.from_numpy(y), torch.tensor(sm_id, dtype=torch.long), torch.from_numpy(fut_cal)




Total samples: 96114


In [6]:
# === 4. Time-aware K-fold with embargo (purged) ===
# === REPLACE: Time-aware K-fold with warm-up & safety ===
def build_time_kfold_splits_safe(
    samples: pd.DataFrame,
    n_splits: int,
    lookback: int,
    horizon: int,
    embargo_days: int,
    verbose: bool = True,
):
    s = samples.sort_values('target_start_date').reset_index(drop=True)

    # 1) 고유 검증 후보 날짜(실제 샘플이 존재하는 날짜만)
    uniq_dates = pd.Series(s['target_start_date'].unique()).sort_values().to_list()

    # 2) warm-up 이후만 검증으로 사용
    warmup = lookback + horizon + embargo_days   # 최소 70일 권장
    earliest_val_date = pd.Timestamp(uniq_dates[0]) + pd.Timedelta(days=warmup)
    val_date_candidates = [d for d in uniq_dates if d >= earliest_val_date]

    if len(val_date_candidates) < n_splits:
        if verbose:
            print(f"[Warn] 검증 후보 날짜가 {len(val_date_candidates)}일 뿐입니다. "
                  f"요청한 n_splits={n_splits} → {len(val_date_candidates)}로 축소.")
        n_splits = max(1, len(val_date_candidates))

    bins = np.array_split(np.array(val_date_candidates), n_splits)

    folds = []
    for k, val_dates in enumerate(bins, start=1):
        if len(val_dates) == 0:
            if verbose: print(f"[Skip] Fold {k}: 빈 검증 구간")
            continue

        val_start = pd.Timestamp(val_dates[0])
        val_end   = pd.Timestamp(val_dates[-1])
        val_mask  = s['target_start_date'].isin(val_dates)
        val_idx   = s.index[val_mask].to_numpy()

        # 학습은 검증 시작일 - embargo 이전의 것만
        cutoff = val_start - pd.Timedelta(days=embargo_days)
        train_mask = s['target_end_date'] < cutoff
        train_idx  = s.index[train_mask].to_numpy()

        if len(train_idx) == 0 or len(val_idx) == 0:
            if verbose:
                print(f"[Skip] Fold {k}: train={len(train_idx)}, val={len(val_idx)} (warm-up/embargo 과도) → 건너뜀")
            continue

        folds.append((train_idx, val_idx))
        if verbose:
            print(f"[Fold {len(folds)}/{n_splits}] "
                  f"Val {val_start.date()}→{val_end.date()} | "
                  f"train_size={len(train_idx):,}, val_size={len(val_idx):,}")

    # 안전장치: 남은 fold가 1개 이하라면 embargo를 완화해서 다시 시도
    if len(folds) <= 1:
        if verbose:
            print("[Info] 유효 fold가 너무 적습니다. embargo를 완화(예: lookback)하여 재시도합니다.")
        relaxed_embargo = max(lookback, embargo_days // 2)  # 최소 lookback
        return build_time_kfold_splits_safe(
            samples, n_splits, lookback, horizon, relaxed_embargo, verbose
        )

    return folds


folds = build_time_kfold_splits_safe(
    samples=samples,
    n_splits=CFG.n_splits,
    lookback=CFG.lookback,
    horizon=CFG.horizon,
    embargo_days=CFG.embargo_days,   # 35
    verbose=True
)


# (Optional) torch Dataset 예시 바인딩
if TORCH_AVAILABLE:
    full_dataset = SalesWindowDataset(
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        lookback=CFG.lookback,
        horizon=CFG.horizon
    )

    # Example: first fold indices
    tr_idx, va_idx = folds[0]
    print(f"First fold -> train:{len(tr_idx)}, val:{len(va_idx)}")


[Fold 1/5] Val 2023-04-09→2023-07-03 | train_size=5,597, val_size=16,598
[Fold 2/5] Val 2023-07-04→2023-09-27 | train_size=22,195, val_size=16,598
[Fold 3/5] Val 2023-09-28→2023-12-22 | train_size=38,793, val_size=16,598
[Fold 4/5] Val 2023-12-23→2024-03-16 | train_size=55,391, val_size=16,405
[Fold 5/5] Val 2024-03-17→2024-06-09 | train_size=71,796, val_size=16,405
First fold -> train:5597, val:16598


In [7]:
# === 6. PatchTST (minimal) + RevIN — PATCHED ===
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class RevIN(nn.Module):
    """Per-sample, per-channel normalization with reversible denorm (stabilized)."""
    def __init__(self, eps: float = 1e-5, min_std: float = 1.0):
        super().__init__()
        self.eps = eps
        self.min_std = min_std  # <--- 추가: 너무 작은 분산 방지

    def forward(self, x, sales_ch: int = 0, stats=None, mode='norm'):
        # x: (B, T, C)
        if mode == 'norm':
            mu = x.mean(dim=1, keepdim=True)                       # (B,1,C)
            sigma = x.std(dim=1, keepdim=True) + self.eps          # (B,1,C)
            sigma = torch.clamp(sigma, min=self.min_std)           # <--- 추가
            x_n = (x - mu) / sigma
            mu_s = mu[:, :, sales_ch]                              # (B,1)
            sg_s = sigma[:, :, sales_ch]                           # (B,1)
            return x_n, (mu_s, sg_s)
        elif mode == 'denorm':
            mu_s, sg_s = stats                                     # (B,1), (B,1)
            y = x                                                  # (B,H)
            return y * sg_s + mu_s
        else:
            raise ValueError("mode must be 'norm' or 'denorm'")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.cos(pos * div)
        pe[:, 1::2] = torch.sin(pos * div)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        P = x.size(1)
        return x + self.pe[:, :P, :]

class PatchTSTMini(nn.Module):
    def __init__(self, lookback, horizon, c_in,
                 d_model=128, n_heads=4, depth=3,
                 patch_len=7, stride=1, dropout=0.1,
                 sales_ch=0, n_store_menu=None, emb_dim=32, fut_feat_dim=None):
        super().__init__()
        self.L, self.H, self.C = lookback, horizon, c_in
        self.patch_len, self.stride, self.sales_ch = patch_len, stride, sales_ch
        self.P = 1 + (self.L - self.patch_len)//self.stride

        self.embed = nn.Linear(self.patch_len*self.C, d_model)
        enc = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers=depth)
        self.posenc = PositionalEncoding(d_model, max_len=self.P)

        # ID embedding
        self.sm_emb = nn.Embedding((n_store_menu or 1)+1, emb_dim)  # 0: UNK

        # Head: last-token + ID embedding
        self.head = nn.Sequential(
            nn.LayerNorm(d_model + emb_dim),
            nn.Linear(d_model + emb_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, self.H)
        )

        # Horizon-wise calendar bias (fut_feat_dim = H*3)
        self.cal_proj = nn.Linear(fut_feat_dim, self.H) if fut_feat_dim else None

        self.revin = RevIN(eps=1e-5, min_std=1.0)

    def patchify(self, x):
        B,L,C = x.shape
        patches=[]
        for st in range(0, L-self.patch_len+1, self.stride):
            p = x[:, st:st+self.patch_len, :].reshape(B, -1)
            patches.append(p)
        return torch.stack(patches, dim=1)

    def forward(self, x, sm_id, fut_cal):
        # RevIN
        x_n, stats = self.revin(x, sales_ch=self.sales_ch, mode='norm')
        # Encoder
        z = self.embed(self.patchify(x_n))
        z = self.encoder(self.posenc(z))
        z = z[:, -1, :]  # last-token pooling

        # concat ID emb
        e = self.sm_emb(sm_id)                     # (B,emb_dim)
        zh = torch.cat([z, e], dim=-1)             # (B,d+emb)

        y_hat_n = self.head(zh)                    # (B,H)
        if self.cal_proj is not None:
            y_hat_n = y_hat_n + self.cal_proj(fut_cal)  # horizon-wise bias

        return y_hat_n, stats



Device: cuda


In [11]:
# === 7. Metrics, Train/Eval, K-fold training — PATCHED ===
from copy import deepcopy
from time import time

def smape_loss_ignore_zero(y_hat, y, eps=1e-6):
    # y_hat,y: (B,H) 원 스케일
    mask = (y != 0).float()
    num = 2.0*torch.abs(y_hat - y)
    den = torch.abs(y_hat) + torch.abs(y) + eps
    s = (num/den) * mask
    return s.sum() / (mask.sum() + 1e-6)


def train_one_fold(
    fold_id: int,
    model_cfg: dict,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    base_df: pd.DataFrame,
    samples_df: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    batch_size: int = 256,
    max_epochs: int = 60,
    lr: float = 5e-4,            # 살짝 낮춤 (denorm MAE 안정화)
    patience: int = 10,
    alpha=0.5,
    mask_zero_in_loss: bool = False   # 필요 시 True
):
    ds_full = SalesWindowDataset(base_df, samples_df, feature_cols, target_col, CFG.lookback, CFG.horizon)
    ds_tr = Subset(ds_full, train_idx); ds_va = Subset(ds_full, val_idx)
    dl_tr = DataLoader(ds_tr, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
    dl_va = DataLoader(ds_va, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

    model = PatchTSTMini(
        lookback=CFG.lookback, horizon=CFG.horizon, c_in=len(feature_cols),
        d_model=model_cfg.get('d_model',192), n_heads=model_cfg.get('n_heads',6),
        depth=model_cfg.get('depth',4), patch_len=CFG.patch_len, stride=CFG.stride,
        dropout=model_cfg.get('dropout',0.2), sales_ch=0,
        n_store_menu=len(SM2ID), emb_dim=32, fut_feat_dim=CFG.horizon*3
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
    L1 = nn.L1Loss(reduction='mean')

    best, best_sd, wait = 1e9, None, 0
    for ep in range(1, max_epochs+1):
        # ---- train
        model.train(); tr_loss=0.0
        for X,y,sm_id,fcal in dl_tr:
            X,y,sm_id,fcal = X.to(device), y.to(device), sm_id.to(device), fcal.to(device)
            opt.zero_grad()
            y_hat_n, stats = model(X, sm_id, fcal)
            y_hat = model.revin(y_hat_n, stats=stats, mode='denorm')
            y_hat = torch.clamp(y_hat, min=0.0)
            loss = alpha*L1(y_hat, y) + (1-alpha)*smape_loss_ignore_zero(y_hat, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item()*X.size(0)
        tr_loss/=len(ds_tr); sch.step()

        # ---- val
        model.eval()
        with torch.no_grad():
            Y=[]; P=[]
            for Xv,yv,sm_idv,fcalv in dl_va:
                Xv,yv,sm_idv,fcalv = Xv.to(device), yv.to(device), sm_idv.to(device), fcalv.to(device)
                y_hat_n, stats = model(Xv, sm_idv, fcalv)
                y_hat = model.revin(y_hat_n, stats=stats, mode='denorm').clamp(min=0.0)
                Y.append(yv); P.append(y_hat)
            Y = torch.cat(Y); P = torch.cat(P)
            val_smape = smape_loss_ignore_zero(P, Y).item()

        print(f"[Fold {fold_id}] Ep{ep:03d} | tr={tr_loss:.4f} | val_sMAPE={val_smape:.4f}")
        if val_smape < best-1e-6:
            best, best_sd, wait = val_smape, deepcopy(model.state_dict()), 0
        else:
            wait+=1
            if wait>=patience:
                print(f"[Fold {fold_id}] Early stop. best sMAPE={best:.4f}")
                break

    torch.save(best_sd, f"./patchtst_fold{fold_id}.pt")
    return best, f"./patchtst_fold{fold_id}.pt"




# === Run K-fold training ===
model_cfg = dict(d_model=128, n_heads=4, depth=3, dropout=0.1)
fold_ckpts = []
fold_scores = []

for k, (tr_idx, va_idx) in enumerate(folds, start=1):
    s, p = train_one_fold(
        fold_id=k,
        model_cfg=model_cfg,
        train_idx=tr_idx,
        val_idx=va_idx,
        base_df=df_feat,
        samples_df=samples,
        feature_cols=FEATURE_COLS,
        target_col=TARGET_COL,
        batch_size=256,
        max_epochs=40,
        lr=1e-3,
        patience=6,
    )
    fold_scores.append(s)
    fold_ckpts.append(p)

print("Fold sMAPE:", fold_scores)
print("Avg sMAPE:", sum(fold_scores)/len(fold_scores))

[Fold 1] Ep001 | tr=2.9725 | val_sMAPE=0.7965
[Fold 1] Ep002 | tr=2.6359 | val_sMAPE=0.7615
[Fold 1] Ep003 | tr=2.5392 | val_sMAPE=0.7888
[Fold 1] Ep004 | tr=2.5021 | val_sMAPE=0.7495
[Fold 1] Ep005 | tr=2.4383 | val_sMAPE=0.7463
[Fold 1] Ep006 | tr=2.3880 | val_sMAPE=0.7475
[Fold 1] Ep007 | tr=2.3060 | val_sMAPE=0.7629
[Fold 1] Ep008 | tr=2.2411 | val_sMAPE=0.7930
[Fold 1] Ep009 | tr=2.2704 | val_sMAPE=0.7899
[Fold 1] Ep010 | tr=2.1758 | val_sMAPE=0.7526
[Fold 1] Ep011 | tr=2.1061 | val_sMAPE=0.7856
[Fold 1] Early stop. best sMAPE=0.7463
[Fold 2] Ep001 | tr=3.0603 | val_sMAPE=0.8146
[Fold 2] Ep002 | tr=2.8092 | val_sMAPE=0.7602
[Fold 2] Ep003 | tr=2.7120 | val_sMAPE=0.7241
[Fold 2] Ep004 | tr=2.6311 | val_sMAPE=0.7050
[Fold 2] Ep005 | tr=2.5550 | val_sMAPE=0.6893
[Fold 2] Ep006 | tr=2.5008 | val_sMAPE=0.6961
[Fold 2] Ep007 | tr=2.4646 | val_sMAPE=0.7605
[Fold 2] Ep008 | tr=2.4063 | val_sMAPE=0.7632
[Fold 2] Ep009 | tr=2.3650 | val_sMAPE=0.7338
[Fold 2] Ep010 | tr=2.3106 | val_sMAPE=0.

In [15]:
# === 8. Inference for TEST files (28->7), fold ensemble — FIXED ===
import glob

# 1) 학습 때 썼던 store_menu ID 맵이 없다면 재구성
try:
    SM2ID  # noqa
except NameError:
    SM_LIST = df_feat['store_menu'].drop_duplicates().tolist()
    SM2ID = {sm: i+1 for i, sm in enumerate(SM_LIST)}  # 0 reserved for UNK
UNK_ID = 0

# 2) 학습 때와 동일한 미래 달력 피처 함수
def future_cal_features(start_date, H=CFG.horizon):
    dates = pd.date_range(start_date, periods=H, freq='D')
    dow = dates.weekday
    dow_sin = np.sin(2*np.pi*dow/7)
    dow_cos = np.cos(2*np.pi*dow/7)
    is_weekend = (dow >= 5).astype(int)
    return np.concatenate([dow_sin, dow_cos, is_weekend], axis=0).astype(np.float32)  # len=H*3

def build_input_tensor_from_block(block_df: pd.DataFrame, feature_cols: List[str]) -> torch.Tensor:
    """ block_df: one store_menu, 28 rows sorted by date """
    g = block_df.sort_values('date')
    X = g[feature_cols].to_numpy(dtype=np.float32)  # (28,C)
    return torch.from_numpy(X).unsqueeze(0)  # (1,28,C)

@torch.no_grad()
def predict_7days_for_testfile(
    test_path: str,
    ckpt_paths: List[str],
    feature_cols: List[str],
    save_path: str
):
    test = pd.read_csv(test_path)
    test['date'] = pd.to_datetime(test['date'])
    if 'store_menu' not in test.columns:
        test['store_menu'] = test['store'].astype(str) + "_" + test['menu'].astype(str)
    test = test.sort_values(['store_menu','date']).reset_index(drop=True)

    # 입력 달력 피처 동일 적용
    test_feat = add_calendar_features(test)

    # Target 날짜 7일
    last_date = test_feat['date'].max()
    target_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=CFG.horizon, freq='D')

    # 3) fold 모델들 로드 — 훈련과 동일 아키텍처 인자 사용!
    models = []
    for ck in ckpt_paths:
        m = PatchTSTMini(
            lookback=CFG.lookback,
            horizon=CFG.horizon,
            c_in=len(feature_cols),
            d_model=model_cfg.get('d_model', 192),   # ← 훈련 때 쓴 값과 동일해야 함
            n_heads=model_cfg.get('n_heads', 6),
            depth=model_cfg.get('depth', 4),
            patch_len=CFG.patch_len,
            stride=CFG.stride,
            dropout=model_cfg.get('dropout', 0.2),
            sales_ch=0,
            n_store_menu=len(SM2ID),                 # ← 반드시 지정
            emb_dim=32,
            fut_feat_dim=CFG.horizon*3               # ← 반드시 지정
        ).to(device)
        sd = torch.load(ck, map_location=device)
        m.load_state_dict(sd, strict=True)           # 구조 일치 강제
        m.eval()
        models.append(m)

    # 4) store_menu별 예측
    preds = {}
    for sm, g in test_feat.groupby('store_menu', sort=False):
        assert len(g) >= CFG.lookback, f"{sm}: need at least {CFG.lookback} rows"
        g_last = g.tail(CFG.lookback)
        X = build_input_tensor_from_block(g_last, feature_cols).to(device)  # (1,28,C)

        # store_menu id 텐서
        sm_id = torch.tensor([SM2ID.get(sm, UNK_ID)], dtype=torch.long, device=device)  # (1,)

        # 미래 달력 피처 텐서 (last_date+1 부터 7일)
        fut_cal = torch.from_numpy(future_cal_features(last_date + pd.Timedelta(days=1), H=CFG.horizon)) \
                    .unsqueeze(0).to(device)  # (1, H*3)

        # fold 앙상블
        fold_outs = []
        for m in models:
            y_hat_n, stats = m(X, sm_id, fut_cal)    # ★ forward 인자 수정
            y_hat = m.revin(y_hat_n, stats=stats, mode='denorm')  # (1,7)
            y_hat = torch.clamp(y_hat, min=0.0)
            fold_outs.append(y_hat)

        y_mean = torch.mean(torch.stack(fold_outs, dim=0), dim=0).squeeze(0).cpu().numpy()  # (7,)
        preds[sm] = y_mean

    # 제출 포맷: rows=7일, cols=store_menu
    sm_list = list(test_feat['store_menu'].drop_duplicates())
    sub = pd.DataFrame(index=target_dates, columns=sm_list, dtype=float)
    for sm in sm_list:
        sub[sm] = preds[sm]

    sub.index.name = 'date'
    sub.to_csv(save_path)
    print(f"[Saved] {save_path} | shape={sub.shape}")
    return sub

# TEST 파일들 일괄 예측
test_files = sorted(glob.glob("./data_filtering/filtered/TEST_0*.csv"))
print("Found test files:", test_files)

for tp in test_files:
    tag = os.path.splitext(os.path.basename(tp))[0]   # e.g., TEST_00
    out_csv = f"./submission_{tag}.csv"
    _ = predict_7days_for_testfile(
        test_path=tp,
        ckpt_paths=fold_ckpts,
        feature_cols=FEATURE_COLS,
        save_path=out_csv
    )


Found test files: ['./data_filtering/filtered/TEST_00.csv', './data_filtering/filtered/TEST_01.csv', './data_filtering/filtered/TEST_02.csv', './data_filtering/filtered/TEST_03.csv', './data_filtering/filtered/TEST_04.csv', './data_filtering/filtered/TEST_05.csv', './data_filtering/filtered/TEST_06.csv', './data_filtering/filtered/TEST_07.csv', './data_filtering/filtered/TEST_08.csv', './data_filtering/filtered/TEST_09.csv']


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_00.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_01.csv | shape=(7, 193)


...

[Saved] ./submission_TEST_07.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_08.csv | shape=(7, 193)


/tmp/ipykernel_419992/1639769317.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ck, map_location=device)


[Saved] ./submission_TEST_09.csv | shape=(7, 193)


In [16]:
import pandas as pd
import glob

# submission_TEST_00.csv ~ submission_TEST_09.csv 파일을 번호 순서대로 불러와서 하나의 데이터프레임으로 합치기
submission_files = sorted(glob.glob("./submission_TEST_0*.csv"), key=lambda x: int(x.split("_")[-1].split(".")[0]))
dfs = []
for file in submission_files:
    df = pd.read_csv(file, index_col=0)
    dfs.append(df)
merged_submission = pd.concat(dfs, axis=0)

merged_submission.to_csv("merged_submission.csv")
print("merged_submission.csv 파일로 저장 완료")


merged_submission.csv 파일로 저장 완료


In [21]:
import pandas as pd

df = pd.read_csv("./result/patchtst_ver2_submission.csv")

In [22]:
df

,영업일자,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
0,TEST_00+1일,7.582977,0.000000,6.383885,3.831066,1.060048,1.269273,0.000000,0.919617,2.085097,...,6.248499,19.124083,17.040487,7.018044,61.413055,38.646084,6.334160,34.926605,9.592996,22.673325
1,TEST_00+2일,0.482706,1.357508,0.719965,0.185127,0.220019,0.332567,1.518661,0.449428,0.457553,...,0.534262,6.461157,3.432558,1.088316,14.514606,7.470022,1.743289,5.758247,1.760395,2.699079
2,TEST_00+3일,3.677402,24.046690,2.042955,1.247549,0.367549,0.626839,5.283661,1.704388,1.351849,...,2.013023,9.775458,5.736273,3.463464,27.066523,19.941452,3.462199,20.887436,4.379932,8.237865
3,TEST_00+4일,4.198968,43.408108,1.826106,1.116944,0.356952,0.534412,8.094390,1.692730,1.566256,...,2.254880,9.312721,7.856285,3.168222,28.539112,20.231974,3.437423,24.140750,4.814998,8.202406
4,TEST_00+5일,4.580680,75.820590,2.064614,1.068065,0.503681,1.109369,19.734537,3.508142,2.068068,...,2.416990,10.144426,8.217239,3.828786,29.251501,20.628690,4.240238,31.534190,5.499929,10.735038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,TEST_09+3일,1.128313,7.912034,3.460363,1.355460,0.595171,0.882040,0.886310,0.074681,1.162825,...,1.838416,9.081733,6.890452,4.503782,66.335440,17.667347,28.107126,45.588573,11.214233,15.303443
66,TEST_09+4일,1.287618,26.022924,2.662077,1.178248,0.583458,0.723141,4.302404,0.111745,1.074248,...,2.077149,8.723890,8.868401,3.649860,63.380330,14.713443,26.634293,45.587770,9.994147,14.047400
67,TEST_09+5일,1.381190,71.843970,3.725892,1.431733,0.698992,1.010277,28.426680,0.153149,1.205716,...,2.549055,9.917093,10.944895,4.670606,71.123210,15.297561,31.813828,55.969420,13.159842,20.061096
68,TEST_09+6일,4.184553,31.051453,8.205604,4.056951,1.394611,2.154738,13.787625,0.578374,2.172318,...,4.115732,14.487218,13.476836,6.775694,114.572350,24.336310,53.631740,59.425545,16.784080,29.610712


In [23]:
def transform_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # 원본 복사
    new_df = df.copy()

    # 숫자 변환할 영역 선택 (1행, 1열 제외)
    numeric_part = new_df.iloc[1:, 1:]

    # 조건 적용: 0 이상 1 이하 → 1, 그 외 값은 반올림
    numeric_part = numeric_part.applymap(
        lambda x: 1 if 0 <= x <= 1 else round(x)
    )

    # 변환된 값 다시 할당
    new_df.iloc[1:, 1:] = numeric_part

    return new_df

result_df = transform_dataframe(df)

/tmp/ipykernel_840594/3440561580.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  numeric_part = numeric_part.applymap(


In [25]:
result_df.to_csv("output33.csv", index=False)